In [1]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [2]:
DATA_DIR = Path("../data/interim/ieee-fraud-detection")
TARGET_COLUMN = "isFraud"
ID_COLUMN = "TransactionID"
TIME_COLUMN = "TransactionDT"

time_col = TIME_COLUMN
target_col = TARGET_COLUMN
train_ratio = 0.8
train_search_width = 0.05
min_test_pos = 200

In [ ]:
df = pd.read_parquet(DATA_DIR / "train.parquet")
df = df.sort_values(time_col)
n = len(df)
y = df[target_col].to_numpy()

In [4]:
train_range = range(
    max(1, int(n * (train_ratio - train_search_width))),
    min(n - 2, int(n * (train_ratio + train_search_width))) + 1
)

best = None
train_end = 0
best_score = float("inf")

for i in tqdm(train_range):
    y_train = y[: i]
    y_test = y[i: ]

    if len(y_test) == 0:
        continue

    n_test_pos = y_test.sum()

    if n_test_pos < min_test_pos:
        continue

    train_target_rate = y_train.mean()
    test_target_rate = y_test.mean()

    score = abs(train_target_rate - test_target_rate)

    if score < best_score:
        best_score = score
        train_end = i
        best = {
            "train_end": i,
            "train_rate": train_target_rate,
            "test_rate": test_target_rate,
            "train_pos": int(y_train.sum()),
            "test_pos": int(n_test_pos),
            "score": score,
        }

best

100%|██████████| 59055/59055 [00:32<00:00, 1820.86it/s]


{'train_end': 498700,
 'train_rate': np.float64(0.035001002606777624),
 'test_rate': np.float64(0.034930313588850175),
 'train_pos': 17455,
 'test_pos': 3208,
 'score': np.float64(7.068901792744997e-05)}

In [37]:
import datetime
df_main = df.iloc[: train_end]
START_DATE = datetime.datetime.strptime('2017-11-30', '%Y-%m-%d')
df_main['DT_M'] = df_main['TransactionDT'].apply(lambda x: (START_DATE + datetime.timedelta(seconds = x)))
df_main['DT_M'] = (df_main['DT_M'].dt.year - 2017) * 12 + df_main['DT_M'].dt.month 

/tmp/ipykernel_1820/1127569060.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_main['DT_M'] = df_main['TransactionDT'].apply(lambda x: (START_DATE + datetime.timedelta(seconds = x)))
/tmp/ipykernel_1820/1127569060.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_main['DT_M'] = (df_main['DT_M'].dt.year - 2017) * 12 + df_main['DT_M'].dt.month


In [38]:
df_main["DT_M"].value_counts()

DT_M
12    137321
15    101632
13     92585
14     86021
16     81141
Name: count, dtype: int64

In [39]:
df_main.groupby('DT_M')['isFraud'].sum()

DT_M
12    3550
13    3705
14    3447
15    4019
16    2734
Name: isFraud, dtype: int64